# Power.log batch simulation runner -- official-Hearthstone-format logs

Each deck cell **re-simulates** the full matrix for that **P1 deck**:
5 P1 agents x 5 P2 agents x 9 P2 decks x 20 games = **4,500 games**,
writing one `.log` per game into `log_power/<agent1>_<deck1>/` plus a `summary.csv`.

* **Resumable**: finished games are skipped, so you can interrupt the kernel and re-run a cell.
* Run cells in any order; each deck is independent.
* **All 9 cells = 40,500 games. Budget ~15-20 h and ~17 GB** (a real Power.log
  is genuinely 200-500 KB per game, so the size is expected -- check your disk first).

## What these logs are

The real Hearthstone client writes `Logs/Power.log` when `log.config` enables the
Power zone. SabberStone already models that exact packet vocabulary: with
`GameConfig.History = true` the engine records a **PowerHistory** of
`CREATE_GAME` / `FULL_ENTITY` / `SHOW_ENTITY` / `HIDE_ENTITY` / `TAG_CHANGE` /
`BLOCK_START` / `BLOCK_END` entries as it plays. So these logs are **not a
reconstruction**: `powerlog_render.py` re-renders the engine's own history --
real trigger-by-trigger ordering, real block nesting -- into the client's text
format.

Verified end to end: every rendered game is parsed back with **hslog**
(HearthSim's parser, the one behind HSReplay.net) and the reconstructed winner,
turn count, hero health, armor and **both boards** are compared against the
SabberStone game object. See `roundtrip_check.py`.

## Things to know before you run

* **These are NEW games.** They are re-simulated, so they do **not** correspond
  to the `log_v2/` games -- do not join the two datasets by game index.
* **Viewpoint = P1's client.** A real Power.log is written by one client, so it
  only holds what that client saw: deck cards are unknown until drawn (yours
  included), and P2's hand/secrets stay hidden until something makes them public.
  This deliberately discards information the simulator has -- that is the point.
* **Labels never appear inside the `.log`.** The agent/playstyle/deck ground
  truth lives in `summary.csv` only, so features and labels stay separate.
* **Timestamps are synthetic**, derived from each game's RNG seed. Real think
  time correlates with which agent is playing, so using it would leak the
  playstyle label into the timing.
* `History=True` costs ~16% runtime and does not bias play (mean turns 14.68 vs
  14.57 over 40 games/arm, Mann-Whitney p=0.96); it does shift the RNG stream,
  so a seed does not reproduce a `History=False` game.
* ~0.8% of games raise on a card SabberStone has not implemented; those are
  recorded as `ERROR` rows in `summary.csv` and skipped.

## Setup

In [15]:
import os, sys
sys.path.insert(0, os.path.abspath("."))   # notebook lives in log_power/
import run_batch_powerlog as pl

print("decks :", ", ".join(pl.DECK_LIST))
print("agents:", ", ".join(pl.AGENT1_FUNCS))
print("games per matchup:", pl.NUM_GAMES,
      " search depth=%d width=%d" % (pl.sim.SEARCH_MAX_DEPTH, pl.sim.SEARCH_MAX_WIDTH))
print("viewpoint: P%d (that client's view only)" % pl.plr.VIEWPOINT)
print("output ->", pl.DEFAULT_LOG_ROOT)

decks : MiraclePirateRogue, ZooDiscardWarlock, RenoKazakusDragonPriest, MidrangeSecretHunter, MidrangeBuffPaladin, MurlocDruid, MidrangeJadeShaman, AggroPirateWarrior, RenoKazakusMage
agents: aggro, control, fatigue, midrange, ramp
games per matchup: 20  search depth=10 width=14
viewpoint: P1 (that client's view only)
output -> d:\test\log_power


## Smoke test first

One validated game per deck (~15 s). Run this before committing to the full
matrix: it exercises all 9 decks and checks every rendered log.

In [16]:
pl.run_smoke()

  OK  aggro      MiraclePirateRogue       vs control    ZooDiscardWarlock        winner=P2   turns=11 lines= 2296   260.2 KB  0.8s
  OK  control    ZooDiscardWarlock        vs midrange   RenoKazakusDragonPriest  winner=P2   turns=17 lines= 2968   345.2 KB  0.7s
  OK  fatigue    RenoKazakusDragonPriest  vs fatigue    MidrangeSecretHunter     winner=P1   turns=20 lines= 3405   385.9 KB  1.1s
  OK  midrange   MidrangeSecretHunter     vs ramp       MidrangeBuffPaladin      winner=P1   turns=17 lines= 3659   418.1 KB  1.0s
  OK  ramp       MidrangeBuffPaladin      vs aggro      MurlocDruid              winner=P1   turns=15 lines= 2684   316.6 KB  0.6s
  OK  aggro      MurlocDruid              vs control    MidrangeJadeShaman       winner=P2   turns=16 lines= 3576   395.3 KB  1.7s
  OK  control    MidrangeJadeShaman       vs midrange   AggroPirateWarrior       winner=P2   turns=15 lines= 3270   381.1 KB  1.0s
  OK  fatigue    AggroPirateWarrior       vs fatigue    RenoKazakusMage          wi

[]

## Preview one game

What a single match looks like in the official format. `CREATE_GAME`, then real
engine blocks -- note the `BLOCK_START BlockType=PLAY` with the ordering the
engine actually executed.

In [17]:
import sim_powerlog as sp, powerlog_render as plr
game, lines, secs = sp.simulate_game_powerlog(
    "aggro", "AggroPirateWarrior", "control", "MurlocDruid", 1, seed=42)
print("%s in %d turns -- %d lines, %.1fs" % (sp.game_result(game)[0],
                                             sp.game_result(game)[3],
                                             len(lines), secs))
print("validate:", plr.validate_lines(lines) or "CLEAN")
print()
for ln in lines[:25]:
    print(ln)
print("...")
i = next(i for i, l in enumerate(lines) if "BlockType=PLAY" in l)
for ln in lines[i:i + 12]:
    print(ln)

P1 in 11 turns -- 2382 lines, 0.6s
validate: CLEAN

D 14:57:53.0000001 GameState.DebugPrintPower() - CREATE_GAME
D 14:57:53.0000001 GameState.DebugPrintPower() -     GameEntity EntityID=1
D 14:57:53.0000001 GameState.DebugPrintPower() -         tag=ZONE value=PLAY
D 14:57:53.0000001 GameState.DebugPrintPower() -         tag=ENTITY_ID value=1
D 14:57:53.0000001 GameState.DebugPrintPower() -         tag=CARDTYPE value=GAME
D 14:57:53.0000001 GameState.DebugPrintPower() -     Player EntityID=2 PlayerID=1 GameAccountId=[hi=144115193835963207 lo=12718623]
D 14:57:53.0000001 GameState.DebugPrintPower() -         tag=CARDTYPE value=PLAYER
D 14:57:53.0000001 GameState.DebugPrintPower() -         tag=MAXHANDSIZE value=10
D 14:57:53.0000001 GameState.DebugPrintPower() -         tag=STARTHANDSIZE value=4
D 14:57:53.0000001 GameState.DebugPrintPower() -         tag=PLAYER_ID value=1
D 14:57:53.0000001 GameState.DebugPrintPower() -         tag=TEAM_ID value=1
D 14:57:53.0000001 GameState.DebugPrint

## Verify against the real Hearthstone parser (optional)

Simulates a few games, renders them, then parses them back with **hslog**
(HearthSim's parser) and diffs the reconstructed state against the engine.

Needs `hslog` **in this notebook's kernel**. This repo has both a global `py -3`
and `d:\test\.venv`, so install it into the interpreter the kernel is actually
running -- the cell prints the exact command if it is missing. Nothing else in
this notebook needs it; skipping only skips the verification.

In [18]:
import sys
try:
    import hslog  # noqa: F401
except ImportError:
    print("hslog is not installed in this kernel -- skipping verification.")
    print('  install with:  "%s" -m pip install hslog' % sys.executable)
else:
    import roundtrip_check as rc
    for i in range(6):
        problems, info = rc.check_game("aggro", "AggroPirateWarrior",
                                       "control", "MurlocDruid", 1, seed=100 + i)
        print("%-5s %s in %2d turns, %5d lines  %s"
              % ("OK" if not problems else "DIFF", info["winner"],
                 info["turns"], info["lines"], "; ".join(problems)))

OK    P2 in 20 turns,  4061 lines  
OK    P2 in 12 turns,  2325 lines  
OK    P2 in 12 turns,  2444 lines  
OK    P2 in 14 turns,  3183 lines  
OK    P2 in 12 turns,  2411 lines  
OK    P2 in 12 turns,  2640 lines  


## P1 deck: MiraclePirateRogue

In [19]:
pl.run_deck1_batch("MiraclePirateRogue")

POWER.LOG BATCH  deck1=MiraclePirateRogue  (4500 games: 5 P1 agents x 5 P2 agents x 9 decks x 20)
  search depth=10 width=14  viewpoint=P1  logs -> d:\test\log_power

### aggro_MiraclePirateRogue  (900 games)
  aggro    vs aggro   /MiraclePirateRogue       P1= 0 P2= 0 D=0 E=0 S=20  [   20/4500   0%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /ZooDiscardWarlock        P1= 0 P2= 0 D=0 E=0 S=20  [   40/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /RenoKazakusDragonPriest  P1= 0 P2= 0 D=0 E=0 S=20  [   60/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeSecretHunter     P1= 0 P2= 0 D=0 E=0 S=20  [   80/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeBuffPaladin      P1= 0 P2= 0 D=0 E=0 S=20  [  100/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MurlocDruid              P1= 0 P2= 0 D=0 E=0 S=20  [  120/4500   3%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeJadeShaman       P1= 0 P2= 0 D=0 E=0 S=20  [  140/4500   3%]  eta   0.

{'done': 4500, 'run': 1, 'secs': 2.4682043999782763, 'errors': 0, 'warned': 0}

## P1 deck: ZooDiscardWarlock

In [20]:
pl.run_deck1_batch("ZooDiscardWarlock")

POWER.LOG BATCH  deck1=ZooDiscardWarlock  (4500 games: 5 P1 agents x 5 P2 agents x 9 decks x 20)
  search depth=10 width=14  viewpoint=P1  logs -> d:\test\log_power

### aggro_ZooDiscardWarlock  (900 games)
  aggro    vs aggro   /MiraclePirateRogue       P1= 0 P2= 0 D=0 E=0 S=20  [   20/4500   0%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /ZooDiscardWarlock        P1= 0 P2= 0 D=0 E=0 S=20  [   40/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /RenoKazakusDragonPriest  P1= 0 P2= 0 D=0 E=0 S=20  [   60/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeSecretHunter     P1= 0 P2= 0 D=0 E=0 S=20  [   80/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeBuffPaladin      P1= 0 P2= 0 D=0 E=0 S=20  [  100/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MurlocDruid              P1= 0 P2= 0 D=0 E=0 S=20  [  120/4500   3%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeJadeShaman       P1= 0 P2= 0 D=0 E=0 S=20  [  140/4500   3%]  eta   0.0 

{'done': 4500, 'run': 3, 'secs': 7.609058500034735, 'errors': 0, 'warned': 0}

## P1 deck: RenoKazakusDragonPriest

In [21]:
pl.run_deck1_batch("RenoKazakusDragonPriest")

POWER.LOG BATCH  deck1=RenoKazakusDragonPriest  (4500 games: 5 P1 agents x 5 P2 agents x 9 decks x 20)
  search depth=10 width=14  viewpoint=P1  logs -> d:\test\log_power

### aggro_RenoKazakusDragonPriest  (900 games)
  aggro    vs aggro   /MiraclePirateRogue       P1= 0 P2= 0 D=0 E=0 S=20  [   20/4500   0%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /ZooDiscardWarlock        P1= 0 P2= 0 D=0 E=0 S=20  [   40/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /RenoKazakusDragonPriest  P1= 0 P2= 0 D=0 E=0 S=20  [   60/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeSecretHunter     P1= 0 P2= 0 D=0 E=0 S=20  [   80/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeBuffPaladin      P1= 0 P2= 0 D=0 E=0 S=20  [  100/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MurlocDruid              P1= 0 P2= 0 D=0 E=0 S=20  [  120/4500   3%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeJadeShaman       P1= 0 P2= 0 D=0 E=0 S=20  [  140/4500   3%]

{'done': 4500, 'run': 3, 'secs': 9.715273100009654, 'errors': 0, 'warned': 0}

## P1 deck: MidrangeSecretHunter

In [22]:
pl.run_deck1_batch("MidrangeSecretHunter")

POWER.LOG BATCH  deck1=MidrangeSecretHunter  (4500 games: 5 P1 agents x 5 P2 agents x 9 decks x 20)
  search depth=10 width=14  viewpoint=P1  logs -> d:\test\log_power

### aggro_MidrangeSecretHunter  (900 games)
  aggro    vs aggro   /MiraclePirateRogue       P1= 0 P2= 0 D=0 E=0 S=20  [   20/4500   0%]  eta   0.0 min  (1.2s)
  aggro    vs aggro   /ZooDiscardWarlock        P1= 0 P2= 0 D=0 E=0 S=20  [   40/4500   1%]  eta   0.0 min  (0.1s)
  aggro    vs aggro   /RenoKazakusDragonPriest  P1= 0 P2= 0 D=0 E=0 S=20  [   60/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeSecretHunter     P1= 0 P2= 0 D=0 E=0 S=20  [   80/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeBuffPaladin      P1= 0 P2= 0 D=0 E=0 S=20  [  100/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MurlocDruid              P1= 0 P2= 0 D=0 E=0 S=20  [  120/4500   3%]  eta   0.0 min  (0.1s)
  aggro    vs aggro   /MidrangeJadeShaman       P1= 0 P2= 0 D=0 E=0 S=20  [  140/4500   3%]  eta 

{'done': 4500, 'run': 0, 'secs': 0.0, 'errors': 0, 'warned': 0}

## P1 deck: MidrangeBuffPaladin

In [23]:
pl.run_deck1_batch("MidrangeBuffPaladin")

POWER.LOG BATCH  deck1=MidrangeBuffPaladin  (4500 games: 5 P1 agents x 5 P2 agents x 9 decks x 20)
  search depth=10 width=14  viewpoint=P1  logs -> d:\test\log_power

### aggro_MidrangeBuffPaladin  (900 games)
  aggro    vs aggro   /MiraclePirateRogue       P1= 0 P2= 0 D=0 E=0 S=20  [   20/4500   0%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /ZooDiscardWarlock        P1= 0 P2= 0 D=0 E=0 S=20  [   40/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /RenoKazakusDragonPriest  P1= 0 P2= 0 D=0 E=0 S=20  [   60/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeSecretHunter     P1= 0 P2= 0 D=0 E=0 S=20  [   80/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeBuffPaladin      P1= 0 P2= 0 D=0 E=0 S=20  [  100/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MurlocDruid              P1= 0 P2= 0 D=0 E=0 S=20  [  120/4500   3%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeJadeShaman       P1= 0 P2= 0 D=0 E=0 S=20  [  140/4500   3%]  eta   

{'done': 4500, 'run': 0, 'secs': 0.0, 'errors': 0, 'warned': 0}

## P1 deck: MurlocDruid

In [24]:
pl.run_deck1_batch("MurlocDruid")

POWER.LOG BATCH  deck1=MurlocDruid  (4500 games: 5 P1 agents x 5 P2 agents x 9 decks x 20)
  search depth=10 width=14  viewpoint=P1  logs -> d:\test\log_power

### aggro_MurlocDruid  (900 games)
  aggro    vs aggro   /MiraclePirateRogue       P1= 0 P2= 0 D=0 E=0 S=20  [   20/4500   0%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /ZooDiscardWarlock        P1= 0 P2= 0 D=0 E=0 S=20  [   40/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /RenoKazakusDragonPriest  P1= 0 P2= 0 D=0 E=0 S=20  [   60/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeSecretHunter     P1= 0 P2= 0 D=0 E=0 S=20  [   80/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeBuffPaladin      P1= 0 P2= 0 D=0 E=0 S=20  [  100/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MurlocDruid              P1= 0 P2= 0 D=0 E=0 S=20  [  120/4500   3%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeJadeShaman       P1= 0 P2= 0 D=0 E=0 S=20  [  140/4500   3%]  eta   0.0 min  (0.0s)


{'done': 4500, 'run': 10, 'secs': 32.39779090002412, 'errors': 0, 'warned': 0}

## P1 deck: MidrangeJadeShaman

In [25]:
pl.run_deck1_batch("MidrangeJadeShaman")

POWER.LOG BATCH  deck1=MidrangeJadeShaman  (4500 games: 5 P1 agents x 5 P2 agents x 9 decks x 20)
  search depth=10 width=14  viewpoint=P1  logs -> d:\test\log_power

### aggro_MidrangeJadeShaman  (900 games)
  aggro    vs aggro   /MiraclePirateRogue       P1= 0 P2= 0 D=0 E=0 S=20  [   20/4500   0%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /ZooDiscardWarlock        P1= 0 P2= 0 D=0 E=0 S=20  [   40/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /RenoKazakusDragonPriest  P1= 0 P2= 0 D=0 E=0 S=20  [   60/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeSecretHunter     P1= 0 P2= 0 D=0 E=0 S=20  [   80/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeBuffPaladin      P1= 0 P2= 0 D=0 E=0 S=20  [  100/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MurlocDruid              P1= 0 P2= 0 D=0 E=0 S=20  [  120/4500   3%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeJadeShaman       P1= 0 P2= 0 D=0 E=0 S=20  [  140/4500   3%]  eta   0.

{'done': 4500, 'run': 11, 'secs': 32.50365639984375, 'errors': 1, 'warned': 0}

## P1 deck: AggroPirateWarrior

In [26]:
pl.run_deck1_batch("AggroPirateWarrior")

POWER.LOG BATCH  deck1=AggroPirateWarrior  (4500 games: 5 P1 agents x 5 P2 agents x 9 decks x 20)
  search depth=10 width=14  viewpoint=P1  logs -> d:\test\log_power

### aggro_AggroPirateWarrior  (900 games)
  aggro    vs aggro   /MiraclePirateRogue       P1= 0 P2= 0 D=0 E=0 S=20  [   20/4500   0%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /ZooDiscardWarlock        P1= 0 P2= 0 D=0 E=0 S=20  [   40/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /RenoKazakusDragonPriest  P1= 0 P2= 0 D=0 E=0 S=20  [   60/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeSecretHunter     P1= 0 P2= 0 D=0 E=0 S=20  [   80/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeBuffPaladin      P1= 0 P2= 0 D=0 E=0 S=20  [  100/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MurlocDruid              P1= 0 P2= 0 D=0 E=0 S=20  [  120/4500   3%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeJadeShaman       P1= 0 P2= 0 D=0 E=0 S=20  [  140/4500   3%]  eta   0.

{'done': 4500, 'run': 6, 'secs': 10.308636799978558, 'errors': 0, 'warned': 0}

## P1 deck: RenoKazakusMage

In [27]:
pl.run_deck1_batch("RenoKazakusMage")

POWER.LOG BATCH  deck1=RenoKazakusMage  (4500 games: 5 P1 agents x 5 P2 agents x 9 decks x 20)
  search depth=10 width=14  viewpoint=P1  logs -> d:\test\log_power

### aggro_RenoKazakusMage  (900 games)
  aggro    vs aggro   /MiraclePirateRogue       P1= 0 P2= 0 D=0 E=0 S=20  [   20/4500   0%]  eta   0.0 min  (0.7s)
  aggro    vs aggro   /ZooDiscardWarlock        P1= 0 P2= 0 D=0 E=0 S=20  [   40/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /RenoKazakusDragonPriest  P1= 0 P2= 0 D=0 E=0 S=20  [   60/4500   1%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeSecretHunter     P1= 0 P2= 0 D=0 E=0 S=20  [   80/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeBuffPaladin      P1= 0 P2= 0 D=0 E=0 S=20  [  100/4500   2%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MurlocDruid              P1= 0 P2= 0 D=0 E=0 S=20  [  120/4500   3%]  eta   0.0 min  (0.0s)
  aggro    vs aggro   /MidrangeJadeShaman       P1= 0 P2= 0 D=0 E=0 S=20  [  140/4500   3%]  eta   0.0 min 

{'done': 4500, 'run': 27, 'secs': 58.050694700097665, 'errors': 2, 'warned': 0}

## Progress overview
Re-run anytime to see how far the batch has come.

In [28]:
import csv, glob, os
root = pl.DEFAULT_LOG_ROOT
total_done = total_err = total_bytes = 0
print("%-40s %8s %8s %9s %8s" % ("directory", "games", "errors", "size MB", "of 900"))
for sp_ in sorted(glob.glob(os.path.join(root, "*", "summary.csv"))):
    with open(sp_, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    err = sum(1 for r in rows if r["winner"] == "ERROR")
    ok = len(rows) - err
    d = os.path.dirname(sp_)
    mb = sum(os.path.getsize(p) for p in glob.glob(os.path.join(d, "*.log"))) / 1e6
    total_done += ok; total_err += err; total_bytes += mb
    print("%-40s %8d %8d %9.1f %8s" % (os.path.basename(d), ok, err, mb,
                                       "done" if ok >= 900 else ""))
print("\nTOTAL: %d games logged, %d errors, %.1f GB  (target 40,500)"
      % (total_done, total_err, total_bytes / 1000.0))

directory                                   games   errors   size MB   of 900
aggro_AggroPirateWarrior                      900        0     294.4     done
aggro_MidrangeBuffPaladin                     900        0     348.9     done
aggro_MidrangeJadeShaman                      900        0     343.8     done
aggro_MidrangeSecretHunter                    900        0     284.1     done
aggro_MiraclePirateRogue                      900        0     308.2     done
aggro_MurlocDruid                             900        5     319.8     done
aggro_RenoKazakusDragonPriest                 900        5     344.6     done
aggro_RenoKazakusMage                         900        2     271.0     done
aggro_ZooDiscardWarlock                       900        0     292.8     done
control_AggroPirateWarrior                    900        0     325.0     done
control_MidrangeBuffPaladin                   900        0     407.4     done
control_MidrangeJadeShaman                    900        4     4